# 06 · Launch Site Geography with Folium

**Question:** does *where* SpaceX launches from relate to landing success, and what do the launch sites have in common geographically?

Steps:
1. Mark every launch site on an interactive map.
2. Add each launch as a green (landed) or red (failed) marker, clustered by site.
3. Measure distances from a launch site to the nearest coastline, railway and city.

> GitHub does not render Folium maps. Open this notebook in [nbviewer](https://nbviewer.org/) or run it locally to interact with the maps; static screenshots are in [`images/`](../images) (`launch_sites_map.png`, `proximity_map.png`).

In [1]:
import folium
import pandas as pd
from folium.features import DivIcon
from folium.plugins import MarkerCluster, MousePosition
from math import atan2, cos, radians, sin, sqrt

In [2]:
spacex_df = pd.read_csv("../data/spacex_launch_geo.csv")[["Launch Site", "Lat", "Long", "class"]]
launch_sites_df = spacex_df.groupby("Launch Site", as_index=False).agg(
    Lat=("Lat", "first"), Long=("Long", "first"), launches=("class", "size"), success_rate=("class", "mean")
)
launch_sites_df

,Launch Site,Lat,Long,launches,success_rate
0,CCAFS LC-40,28.562302,-80.577356,26,0.269231
1,CCAFS SLC-40,28.563197,-80.576820,7,0.428571
2,KSC LC-39A,28.573255,-80.646895,13,0.769231
3,VAFB SLC-4E,34.632834,-120.610745,10,0.400000


## 1. All launch sites on a map

Every site sits close to a coastline and in the southern half of the US, as close to the Equator as US territory allows. Launching eastward near the Equator gets extra speed from Earth's rotation, and launching over the ocean keeps debris away from people.

In [3]:
def label(text, color="#d35400"):
    return DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                   html=f'<div style="font-size:12px;color:{color};white-space:nowrap;"><b>{text}</b></div>')

site_map = folium.Map(location=[29.559684888503615, -95.0830971930759], zoom_start=4)
for _, row in launch_sites_df.iterrows():
    coord = [row["Lat"], row["Long"]]
    folium.Circle(coord, radius=1000, color="#d35400", fill=True, popup=row["Launch Site"]).add_to(site_map)
    folium.Marker(coord, icon=label(row["Launch Site"])).add_to(site_map)
site_map

## 2. Launch outcomes by site

Green markers are successful first-stage landings, red markers are failures. Click a cluster to expand it.

In [4]:
spacex_df["marker_color"] = spacex_df["class"].map({1: "green", 0: "red"})

marker_cluster = MarkerCluster().add_to(site_map)
for _, row in spacex_df.iterrows():
    folium.Marker(
        [row["Lat"], row["Long"]],
        icon=folium.Icon(color="white", icon_color=row["marker_color"]),
        popup=f'{row["Launch Site"]}: {"landed" if row["class"] else "failed"}',
    ).add_to(marker_cluster)
site_map

In [5]:
launch_sites_df.sort_values("success_rate", ascending=False).style.format({"success_rate": "{:.0%}"})

,Launch Site,Lat,Long,launches,success_rate
2,KSC LC-39A,28.573255,-80.646895,13,77%
1,CCAFS SLC-40,28.563197,-80.576820,7,43%
3,VAFB SLC-4E,34.632834,-120.610745,10,40%
0,CCAFS LC-40,28.562302,-80.577356,26,27%


**KSC LC-39A** has the highest landing success rate and **CCAFS LC-40** has the most launches, including many of the early experimental landing attempts that failed.

## 3. Distances from CCAFS SLC-40 to nearby features

The coordinates of each feature were read off the map using `MousePosition`, so they are approximate.

In [6]:
def calculate_distance(lat1, lon1, lat2, lon2):
    """Great-circle distance in km (haversine formula)."""
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, (lat1, lon1, lat2, lon2))
    a = sin((lat2 - lat1) / 2) ** 2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

site = launch_sites_df.set_index("Launch Site").loc["CCAFS SLC-40"]
site_coord = [site["Lat"], site["Long"]]

proximities = {
    "Coastline": [28.56342, -80.56794],
    "Railway": [28.57208, -80.58527],
    "City (Titusville)": [28.61208, -80.80764],
}

prox_map = folium.Map(location=site_coord, zoom_start=14)
prox_map.add_child(MousePosition(position="topright", separator=" Long: ", prefix="Lat:", num_digits=5))
folium.Marker(site_coord, popup="CCAFS SLC-40", icon=folium.Icon(color="blue", icon="rocket", prefix="fa")).add_to(prox_map)

rows = []
for name, coord in proximities.items():
    distance = calculate_distance(*site_coord, *coord)
    rows.append({"Feature": name, "Distance (km)": round(distance, 2)})
    folium.Marker(coord, icon=label(f"{name.split(' (')[0]}: {distance:.1f} km")).add_to(prox_map)
    folium.PolyLine([site_coord, coord], weight=2).add_to(prox_map)

display(pd.DataFrame(rows))
prox_map

,Feature,Distance (km)
0,Coastline,0.87
1,Railway,1.29
2,City (Titusville),23.19


## Findings

- **Coastline, about 1 km:** rockets fly out over the ocean, and failed boosters or debris fall into water.
- **Railway, about 1.3 km:** heavy hardware and propellant need a transport connection.
- **City, over 20 km:** populated areas are kept at a safe distance.
- **Near the Equator and on the east coast:** eastward launches get extra speed from Earth's rotation. The West Coast site (VAFB) serves polar orbits.
- Success rates differ by site, so **launch site is a useful feature** for the model in notebook 07.